In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Configuration
SILVER_SCHEMA = "workspace.silver"
GOLD_SCHEMA = "workspace.gold"

# Read silver layer tables
fact_transactions = spark.table(f"{SILVER_SCHEMA}.fact_transactions")
dim_customers = spark.table(f"{SILVER_SCHEMA}.dim_customers")
dim_products = spark.table(f"{SILVER_SCHEMA}.dim_products")
dim_stores = spark.table(f"{SILVER_SCHEMA}.dim_stores")

print("✓ Silver layer tables loaded successfully")

In [0]:
# Calculate customer lifetime value and behavior metrics
gold_customer_ltv = (
    fact_transactions
    .join(dim_customers, "customer_id")
    .groupBy(
        "customer_id",
        "customer_full_name",
        "customer_email",
        "customer_city",
        "customer_province",
        "customer_segment",
        "customer_signup_date"
    )
    .agg(
        F.count("transaction_id").alias("total_transactions"),
        F.sum("total_amount").alias("lifetime_value"),
        F.avg("total_amount").alias("avg_transaction_value"),
        F.sum("quantity").alias("total_items_purchased"),
        F.min("transaction_date").alias("first_purchase_date"),
        F.max("transaction_date").alias("last_purchase_date"),
        F.countDistinct("product_id").alias("unique_products_purchased"),
        F.countDistinct("store_id").alias("unique_stores_visited")
    )
    .withColumn(
        "days_since_signup",
        F.datediff(F.col("last_purchase_date"), F.col("customer_signup_date"))
    )
    .withColumn(
        "days_since_last_purchase",
        F.datediff(F.current_date(), F.col("last_purchase_date"))
    )
    .withColumn(
        "purchase_frequency",
        F.round(F.col("total_transactions") / (F.col("days_since_signup") + 1) * 30, 2)
    )
    .withColumn("created_at", F.current_timestamp())
)

# Add customer value tier
gold_customer_ltv = (
    gold_customer_ltv
    .withColumn(
        "customer_tier",
        F.when(F.col("lifetime_value") >= 10000, "Platinum")
        .when(F.col("lifetime_value") >= 5000, "Gold")
        .when(F.col("lifetime_value") >= 2000, "Silver")
        .otherwise("Bronze")
    )
)

print(f"Customer LTV records: {gold_customer_ltv.count():,}")
display(gold_customer_ltv.orderBy(F.desc("lifetime_value")).limit(10))

In [0]:
# RFM (Recency, Frequency, Monetary) segmentation
current_date = F.current_date()

gold_customer_rfm = (
    fact_transactions
    .join(dim_customers, "customer_id")
    .groupBy(
        "customer_id",
        "customer_full_name",
        "customer_segment"
    )
    .agg(
        F.max("transaction_date").alias("last_purchase_date"),
        F.count("transaction_id").alias("frequency"),
        F.sum("total_amount").alias("monetary_value")
    )
    .withColumn("recency_days", F.datediff(current_date, F.col("last_purchase_date")))
)

# Calculate RFM scores (1-5 scale)
recency_window = Window.orderBy("recency_days")
frequency_window = Window.orderBy(F.desc("frequency"))
monetary_window = Window.orderBy(F.desc("monetary_value"))

gold_customer_rfm = (
    gold_customer_rfm
    .withColumn("r_score", F.ntile(5).over(recency_window))
    .withColumn("f_score", F.ntile(5).over(frequency_window))
    .withColumn("m_score", F.ntile(5).over(monetary_window))
    .withColumn("rfm_score", F.col("r_score") + F.col("f_score") + F.col("m_score"))
)

# Assign customer segments based on RFM
gold_customer_rfm = (
    gold_customer_rfm
    .withColumn(
        "rfm_segment",
        F.when((F.col("r_score") >= 4) & (F.col("f_score") >= 4) & (F.col("m_score") >= 4), "Champions")
        .when((F.col("r_score") >= 3) & (F.col("f_score") >= 3) & (F.col("m_score") >= 3), "Loyal Customers")
        .when((F.col("r_score") >= 4) & (F.col("f_score") <= 2), "New Customers")
        .when((F.col("r_score") <= 2) & (F.col("f_score") >= 3), "At Risk")
        .when((F.col("r_score") <= 2) & (F.col("f_score") <= 2), "Lost Customers")
        .otherwise("Potential Loyalists")
    )
    .withColumn("created_at", F.current_timestamp())
)

print(f"RFM segmentation records: {gold_customer_rfm.count():,}")
display(gold_customer_rfm.orderBy(F.desc("rfm_score")).limit(10))

In [0]:
# Monthly cohort analysis
gold_customer_cohorts = (
    fact_transactions
    .join(dim_customers, "customer_id")
    .withColumn("cohort_month", F.date_format(F.col("customer_signup_date"), "yyyy-MM"))
    .withColumn("transaction_month", F.date_format(F.col("transaction_date"), "yyyy-MM"))
    .groupBy("cohort_month", "transaction_month")
    .agg(
        F.countDistinct("customer_id").alias("active_customers"),
        F.sum("total_amount").alias("cohort_revenue"),
        F.avg("total_amount").alias("avg_revenue_per_transaction")
    )
    .withColumn("created_at", F.current_timestamp())
)

# Calculate cohort size (first month customers)
cohort_sizes = (
    gold_customer_cohorts
    .filter(F.col("cohort_month") == F.col("transaction_month"))
    .select(
        F.col("cohort_month").alias("cohort"),
        F.col("active_customers").alias("cohort_size")
    )
)

# Calculate retention rates
gold_customer_cohorts = (
    gold_customer_cohorts
    .join(cohort_sizes, gold_customer_cohorts.cohort_month == cohort_sizes.cohort)
    .withColumn(
        "retention_rate",
        F.round(F.col("active_customers") / F.col("cohort_size") * 100, 2)
    )
    .drop("cohort")
)

print(f"Cohort analysis records: {gold_customer_cohorts.count():,}")
display(gold_customer_cohorts.orderBy(F.desc("cohort_month"), "transaction_month").limit(15))

In [0]:
# Identify customers at risk of churning
gold_customer_churn_risk = (
    fact_transactions
    .join(dim_customers, "customer_id")
    .groupBy(
        "customer_id",
        "customer_full_name",
        "customer_email",
        "customer_segment"
    )
    .agg(
        F.max("transaction_date").alias("last_purchase_date"),
        F.count("transaction_id").alias("total_transactions"),
        F.sum("total_amount").alias("lifetime_value"),
        F.avg("total_amount").alias("avg_order_value")
    )
    .withColumn("days_since_last_purchase", F.datediff(F.current_date(), F.col("last_purchase_date")))
    .withColumn(
        "churn_risk",
        F.when(F.col("days_since_last_purchase") > 90, "High")
        .when(F.col("days_since_last_purchase") > 60, "Medium")
        .when(F.col("days_since_last_purchase") > 30, "Low")
        .otherwise("Active")
    )
    .withColumn(
        "recommended_action",
        F.when(F.col("churn_risk") == "High", "Win-back campaign")
        .when(F.col("churn_risk") == "Medium", "Re-engagement email")
        .when(F.col("churn_risk") == "Low", "Promotional offer")
        .otherwise("Continue nurturing")
    )
    .withColumn("created_at", F.current_timestamp())
)

print(f"Churn risk records: {gold_customer_churn_risk.count():,}")
print("\nChurn risk distribution:")
display(gold_customer_churn_risk.groupBy("churn_risk").count().orderBy("churn_risk"))

In [0]:
# Write all customer analytics tables
gold_customer_ltv.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.customer_lifetime_value")
print("✓ Wrote customer_lifetime_value")

gold_customer_rfm.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.customer_rfm_segmentation")
print("✓ Wrote customer_rfm_segmentation")

gold_customer_cohorts.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.customer_cohorts")
print("✓ Wrote customer_cohorts")

gold_customer_churn_risk.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.customer_churn_risk")
print("✓ Wrote customer_churn_risk")

print("\n" + "="*50)
print("Customer Analytics Gold Layer - COMPLETED")
print("="*50)